In [1]:
from pybm.examples.predator_prey import  generate_synthetic_data
import torch


true_const_values = {
    "growth_rate_prey": 0.5,
    "growth_rate_predator": -0.2,
    "predation_rate": 0.02,
    "conversion_efficiency": 0.1
}

# Generate model and data from /home/urhp/Documents/PyBM/src/pybm/examples/predator_prey.py
model, components, times = generate_synthetic_data()
model.engine = "torch"
print(str(model))
# send to torch 
for var_name, var in model.vars.items():
    if var.data is not None:
        var.data.to_torch()
# change times to a torch tensor
t_eval = torch.tensor(times)
# reshape into (1, n_times)
#t_eval = torch.reshape(t_eval, (1, -1))

# print initial values of the variables
print("Initial values of the variables:")
for var_name, var in model.vars.items():
    print(f"{var_name}: {var.initial}")  

# set initial values of the constants to bad guesses
model.consts["growth_rate_prey"].initial_value = 0.1 #0.5
model.consts["growth_rate_predator"].initial_value = 0.1 #-0.2
model.consts["predation_rate"].initial_value = 0.1 #0.02
model.consts["conversion_efficiency"].initial_value = 0.1 #0.1



# print initial values of the constants
print("\nInitial values of the constants:")
for const_name, const in model.consts.items():
    print(f"{const_name}: {const.initial_value}")


Model(Entities: [],
 Vars: ['n_prey', 'n_predator', 'temperature'],
 Consts: ['growth_rate_prey', 'growth_rate_predator', 'predation_rate', 'conversion_efficiency'])
Initial values of the variables:
n_prey: 40.000492061342996
n_predator: 8.016754323781468
temperature: None

Initial values of the constants:
growth_rate_prey: 0.1
growth_rate_predator: 0.1
predation_rate: 0.1
conversion_efficiency: 0.1


In [2]:
from pybm.estimate.gradient_matching import estimate_gradient_matching
from pybm.estimate.multishooting_torch import estimate_torch


# n_subintervals=1 would be plain single-shooting -- no matching condition
# to enforce, so the "multishooting" part of this pipeline would do
# nothing. Use several segments so the fit actually benefits from it.
n_subintervals = 10

# estimate the parameters using gradient matching (fast, but less accurate)
# to get a good initial guess for the parameters
result = estimate_gradient_matching(model, t_eval)
init_params = result.init_params(n_subintervals=n_subintervals)
init_consts = result.consts
print("\n Initial guesses, collected from gradient matching:")
for const_name, const in model.consts.items():
    print(f"{const_name}: {init_consts[const.index_in_ctx]} (real value: {true_const_values[const_name]})")



 Initial guesses, collected from gradient matching:
growth_rate_prey: 0.4995117555895066 (real value: 0.5)
growth_rate_predator: -0.19993410872528386 (real value: -0.2)
predation_rate: 0.019927213658000167 (real value: 0.02)
conversion_efficiency: 0.1000485050022417 (real value: 0.1)


In [ ]:
import importlib
import matplotlib.pyplot as plt
import pybm.estimate.analysis_torch as analysis_torch_module

analysis_torch_module = importlib.reload(analysis_torch_module)
simulate_multishooting_torch = analysis_torch_module.simulate_multishooting
MSE = analysis_torch_module.MSE

# Reconstruct the fitted multishooting trajectory from the full parameter vector.
# n_subintervals must match what was actually used to fit `fit.x` above.
fitted_solution = simulate_multishooting_torch(
    model,
    t_eval,
    params=init_params,
    n_subintervals=n_subintervals,
 )

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)

series = [
    ("n_prey", "Prey"),
    ("n_predator", "Predators"),
]

for index, (name, title) in enumerate(series):
    ax = axes[index]
    observed = components[name].data.x

    ax.plot(
        times, observed,
        label="Observed trajectory",
        color="black",
        alpha=0.7,
    )
    ax.plot(
        times, fitted_solution.y[index],
        label="Torch multishooting reconstruction",
        color="tab:blue",
        linewidth=2,
    )

    ax.set_title(title)
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Population")
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle("Observed and Torch multishooting reconstructed trajectories")
fig.tight_layout()
plt.show()

# MSE of trajectory reconstruction (data-reseeded segments, not the fitted
# shooting seeds -- this scores `init_consts` alone).
# n_subintervals=1 (single-shooting, harshest) vs. n_subintervals=n_subintervals
# (as forgiving as the fit above).
print(f"MSE (gradient-matching guess), single-shooting: {MSE(model, t_eval, n_subintervals=1, consts=init_consts):.4g}")
print(f"MSE (gradient-matching guess), n_subintervals={n_subintervals}: {MSE(model, t_eval, n_subintervals=n_subintervals, consts=init_consts):.4g}")


In [ ]:
# estimate the parameters using multishooting (slower, but more accurate)
# max_iter/gtol are now shared between "constraints" and "weighted_sum"
# (previously separate n_iter/maxiter -- that param no longer exists).
fit = estimate_torch(model, t_eval, method="constraints",
                      n_subintervals=n_subintervals, n_candidates=1,
                      init_params=init_params, verbose=2, max_iter=200, gtol=1e-4, patience=5)
const_values_torch = fit.consts

| niter |f evals|CG iter|  obj func   |tr radius |   opt    |  c viol  |
|-------|-------|-------|-------------|----------|----------|----------|
|   1   |   1   |   0   | +1.7772e+00 | 1.00e+00 | 7.10e+01 | 7.18e-01 |
|   2   |   3   |   1   | +1.7772e+00 | 1.00e-01 | 7.10e+01 | 7.18e-01 |
|   3   |   4   |   3   | +1.6126e+00 | 1.00e-01 | 5.90e+02 | 3.60e-01 |
|   4   |   5   |   6   | +1.6126e+00 | 1.00e-02 | 5.90e+02 | 3.60e-01 |
|   5   |   6   |   9   | +1.6126e+00 | 1.00e-03 | 5.90e+02 | 3.60e-01 |
|   6   |   7   |  12   | +1.6126e+00 | 1.00e-04 | 5.90e+02 | 3.60e-01 |
|   7   |   8   |  15   | +1.6182e+00 | 5.64e-04 | 9.24e+01 | 3.59e-01 |
|   8   |   9   |  17   | +1.6182e+00 | 5.64e-05 | 9.24e+01 | 3.59e-01 |
|   9   |  10   |  19   | +1.6182e+00 | 5.64e-06 | 9.24e+01 | 3.59e-01 |
|  10   |  11   |  21   | +1.6144e+00 | 3.95e-05 | 1.40e+02 | 3.58e-01 |
|  11   |  12   |  22   | +1.6144e+00 | 3.95e-06 | 1.40e+02 | 3.58e-01 |
|  12   |  13   |  23   | +1.6144e+00 | 1.97e-06 | 

In [5]:
# update model constants with estimated values
for const_name, const in model.consts.items():
    const.initial_value = const_values_torch[const.index_in_ctx].item()

print("\nEstimated values of the constants:")
for const_name, const in model.consts.items():
    print(f"{const_name}: {const.initial_value}, true value: {true_const_values[const_name]}")


Estimated values of the constants:
growth_rate_prey: 0.4996239436147203, true value: 0.5
growth_rate_predator: -0.19997514069845787, true value: -0.2
predation_rate: 0.02004089859975838, true value: 0.02
conversion_efficiency: 0.09982406020059381, true value: 0.1


In [ ]:
import importlib
import matplotlib.pyplot as plt
import pybm.estimate.analysis_torch as analysis_torch_module

analysis_torch_module = importlib.reload(analysis_torch_module)
simulate_multishooting_torch = analysis_torch_module.simulate_multishooting
MSE = analysis_torch_module.MSE

# Reconstruct the fitted multishooting trajectory from the full parameter vector.
# n_subintervals must match what was actually used to fit `fit.x` above.
fitted_solution = simulate_multishooting_torch(
    model,
    t_eval,
    params=fit.x,
    n_subintervals=n_subintervals,
 )

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)

series = [
    ("n_prey", "Prey"),
    ("n_predator", "Predators"),
]

for index, (name, title) in enumerate(series):
    ax = axes[index]
    observed = components[name].data.x

    ax.plot(
        times, observed,
        label="Observed trajectory",
        color="black",
        alpha=0.7,
    )
    ax.plot(
        times, fitted_solution.y[index],
        label="Torch multishooting reconstruction",
        color="tab:blue",
        linewidth=2,
    )

    ax.set_title(title)
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Population")
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle("Observed and Torch multishooting reconstructed trajectories")
fig.tight_layout()
plt.show()

# MSE of trajectory reconstruction (data-reseeded segments, not the fitted
# shooting seeds -- this scores `fit.consts` alone).
# n_subintervals=1 (single-shooting, harshest) vs. n_subintervals=n_subintervals
# (as forgiving as the fit above) -- compare against the gradient-matching-guess
# numbers printed in the previous cell.
print(f"MSE (fitted), single-shooting: {MSE(model, t_eval, n_subintervals=1, consts=fit.consts):.4g}")
print(f"MSE (fitted), n_subintervals={n_subintervals}: {MSE(model, t_eval, n_subintervals=n_subintervals, consts=fit.consts):.4g}")
